# Winston prompt + memory diagnostics lab

**Goal:** inspect why Winston sometimes loses memory, what actually lands in the
final prompt for a given turn, and how phrasing drifts when page context changes.

This notebook is a *lab*, not a dashboard. You are expected to:

1. Run Section 1 once to load data.
2. Jump to whichever section matches the question you're asking.
3. Edit the variables at the top of each experiment cell and rerun.

It works in three modes, in this order of preference:

| Mode | When used | Requires |
|---|---|---|
| CSV | Default. Drop exports into `./data/` | `ai_conversations_rows.csv` etc. |
| Live DB | Set `prefer_db=True` in Section 1 | `DATABASE_URL` env var + `psycopg` or `psycopg2` |
| Synthetic | Fallback if nothing is available | Nothing — runs out of the box |

All heavy logic lives in `prompt_memory_diagnostics.py` next to this notebook.
The notebook's job is to *call* the helpers and *show* results.


## Section 1 — Setup

Imports, path config, data load. Edit `DATA_DIR` if your CSV exports live somewhere else.
Set `PREFER_DB = True` to try the live DB first.

In [1]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make this notebook portable: ensure the helper module is importable whether
# you launched Jupyter from the repo root or from notebooks/.
NB_DIR = Path.cwd()
if (NB_DIR / "prompt_memory_diagnostics.py").exists():
    sys.path.insert(0, str(NB_DIR))
elif (NB_DIR / "notebooks" / "prompt_memory_diagnostics.py").exists():
    sys.path.insert(0, str(NB_DIR / "notebooks"))

import prompt_memory_diagnostics as pmd

# ---- Config you'll actually edit -------------------------------------------
DATA_DIR = Path("./data")           # where ai_*_rows.csv live
PREFER_DB = False                    # True = try DATABASE_URL first
TOKEN_BUDGET_DEFAULT = 16000         # used for classification heuristics
# ---------------------------------------------------------------------------

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 140)


In [2]:
data = pmd.load_data(data_dir=DATA_DIR, prefer_db=PREFER_DB)

print(f"source: {data.source}")
for note in data.notes:
    print(f"  - {note}")
print()
print("rows:")
print(f"  conversations    {len(data.conversations):>8,}")
print(f"  messages         {len(data.messages):>8,}")
print(f"  gateway_logs     {len(data.gateway_logs):>8,}")
print(f"  prompt_receipts  {len(data.prompt_receipts):>8,}")


source: synthetic
  - no CSVs or DB found — using synthetic demo data

rows:
  conversations           2
  messages               10
  gateway_logs            5
  prompt_receipts         0


## Section 2 — Data load and profiling

Quick health summary: how much of the data is linked, how big conversations are,
and the shape of the prompt-token distribution.

In [3]:
summary = pmd.profile(data)
summary


{'source': 'synthetic',
 'notes': ['no CSVs or DB found — using synthetic demo data'],
 'counts': {'conversations': 2,
  'messages': 10,
  'gateway_logs': 5,
  'prompt_receipts': 0},
 'logs_linked_to_conversation_pct': 100.0,
 'avg_messages_per_conversation': np.float64(5.0),
 'longest_conversations': {'c-001': 6, 'c-002': 4},
 'prompt_token_p50': 1520,
 'prompt_token_p95': 1808,
 'prompt_token_max': 1840}

In [ ]:
# Null rates for important columns (messy CSVs can hide gaps here)
def null_rates(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols or df.empty:
        return pd.Series(dtype=float)
    return df[cols].isna().mean().round(3).sort_values(ascending=False)

print("conversations:")
print(null_rates(data.conversations, ["scope_type", "scope_label", "context_summary", "thread_kind", "last_route"]))
print()
print("gateway_logs:")
print(null_rates(data.gateway_logs, ["conversation_id", "prompt_audit_json", "timings_json", "matched_pattern", "route_lane"]))
print()
print("messages:")
print(null_rates(data.messages, ["tool_calls", "citations", "token_count"]))


In [1]:
# Longest conversations and highest-token turns (where memory issues usually show up)
longest = (
    data.messages.groupby("conversation_id").size().sort_values(ascending=False).head(10)
    if not data.messages.empty and "conversation_id" in data.messages.columns
    else pd.Series(dtype=int)
)
print("Longest conversations (by message count):")
print(longest.to_string())
print()
if not data.gateway_logs.empty and "prompt_tokens" in data.gateway_logs.columns:
    hot = data.gateway_logs.assign(
        prompt_tokens=pd.to_numeric(data.gateway_logs["prompt_tokens"], errors="coerce")
    ).nlargest(10, "prompt_tokens")[
        ["conversation_id", "route_lane", "route_model", "prompt_tokens", "rag_chunks_used"]
    ]
    print("Highest-token turns:")
    print(hot.to_string(index=False))


NameError: name 'data' is not defined

## Section 3 — Conversation explorer

Pick a conversation and see its metadata, transcript, linked gateway logs, and
(if available) prompt receipts side by side.

In [ ]:
# Browse the 25 most recent conversations
pmd.list_conversations(data, limit=25)


In [ ]:
# ---- Edit this and rerun --------------------------------------------------
CONVERSATION_ID = None  # e.g. "c-001" — None picks the most recent
# ---------------------------------------------------------------------------

if CONVERSATION_ID is None:
    convs = pmd.list_conversations(data, limit=1)
    CONVERSATION_ID = (
        convs.iloc[0]["conversation_id"] if not convs.empty else None
    )
print(f"inspecting: {CONVERSATION_ID}")

bundle = pmd.conversation_bundle(data, CONVERSATION_ID) if CONVERSATION_ID else {}
bundle.get("conversation")


In [ ]:
print(pmd.render_transcript(bundle, preview_chars=400))


In [ ]:
logs_slice = bundle.get("logs", pd.DataFrame())
if logs_slice.empty:
    print("(no gateway logs for this conversation)")
else:
    cols = [
        c for c in [
            "created_at", "route_lane", "route_model", "prompt_tokens",
            "completion_tokens", "rag_chunks_used", "fallback_used",
            "elapsed_ms", "matched_pattern",
        ] if c in logs_slice.columns
    ]
    display(logs_slice[cols])


## Section 4 — Turn-by-turn prompt inspector

For the selected conversation, pick a turn index (0-based) and inspect what the
gateway actually assembled — using `prompt_audit_json` if the new instrumentation
is present, otherwise the aggregate token count.

This section is about real turns. Section 5 is the *simulator* where you can
override context and see how assembly would change.

In [ ]:
# ---- Edit this and rerun --------------------------------------------------
TURN_INDEX = 0  # 0 = oldest turn in this conversation, -1 = most recent
# ---------------------------------------------------------------------------

logs_slice = bundle.get("logs", pd.DataFrame())
if logs_slice.empty:
    print("(no logs for this conversation — skip to Section 5)")
else:
    row = logs_slice.iloc[TURN_INDEX]
    audit = pmd.extract_prompt_audit(row)
    print(pmd.render_prompt_audit(audit))
    print()
    print("user message preview:")
    print("  " + pmd.truncate_preview(row.get("message_preview"), 400))


In [ ]:
# If you have prompt_receipts exported, render the actual stored prompt sections.
# This table is new instrumentation — it's optional.
receipts_slice = bundle.get("receipts", pd.DataFrame())
if receipts_slice.empty:
    print("(no prompt_receipts for this conversation — stored prompt sections unavailable)")
else:
    for i, row in receipts_slice.iterrows():
        print(f"=== receipt {i} ===")
        for key in [
            "system_prompt", "context_block", "visible_context",
            "rag_block", "current_user_message",
        ]:
            val = row.get(key)
            if pd.isna(val) or val is None or val == "":
                continue
            print(pmd.boxed(key.upper(), pmd.truncate_preview(str(val), 800)))


## Section 5 — Manual context override lab ⭐

This is the main lab. Edit the `inputs` block below and rerun to see exactly
how the final prompt would be assembled, what the token profile looks like,
and what would be trimmed if the budget were exceeded.

**Tip:** the quickest way to test phrasing drift is to hold everything
constant and change only `page_name` or `assistant_scope_label`. Section 7
automates those sweeps; this section is for one-off exploration.

All variables map 1:1 to `ContextInputs` in the helper module.

In [ ]:
# Start from a sensible Meridian fund-detail baseline, then tweak.
inputs = pmd.default_lab_inputs()

# ---- Edit these and rerun -------------------------------------------------
inputs.page_name = "Fund Detail"
inputs.module_name = "REPE"
inputs.environment_name = "Meridian Capital"
inputs.business_name = "Meridian Capital Partners"

inputs.assistant_scope_type = "fund"
inputs.assistant_scope_id = "fund-mvf1"
inputs.assistant_scope_label = "Meridian Value Fund I"

inputs.visible_context_summary = (
    "User is on the Fund Detail page for Meridian Value Fund I. "
    "Visible widgets: TVPI 1.42x, Net IRR 11.8%, 3 assets, 2025Q4 snapshot."
)
inputs.thread_entity_state = {
    "fund_id": "fund-mvf1",
    "fund_name": "Meridian Value Fund I",
    "quarter": "2025Q4",
    "assets_in_view": 3,
}

inputs.current_user_message = "What is going on here?"
inputs.prior_history_messages = [
    {"role": "user", "content": "What is the current TVPI for this fund?"},
    {"role": "assistant", "content": "TVPI is 1.42x as of 2025-Q4."},
    {"role": "user", "content": "And the IRR?"},
    {"role": "assistant", "content": "Net IRR 11.8%, gross IRR 14.2%."},
]
inputs.rag_snippets = [
    "Meridian Value Fund I 2025Q4 quarter-close: TVPI 1.42, DPI 0.35, RVPI 1.07.",
    "Glossary: TVPI = (cumulative distributions + NAV) / contributed capital.",
]

inputs.lane = "B"
inputs.model = "gpt-5-mini"
inputs.assistant_role = "analyst"
inputs.include_rag = True
inputs.include_visible_context = True
inputs.include_thread_summary = True
inputs.history_depth = None            # None = all history
inputs.simulated_max_prompt_tokens = 16000
# ---------------------------------------------------------------------------

ap = pmd.assemble_prompt(inputs)
print(pmd.render_assembled(ap))


In [ ]:
# Dominant section quick check
totals = pd.Series(ap.section_tokens).sort_values(ascending=False)
display(totals.to_frame("tokens"))
print(f"dominant section: {totals.index[0]} ({totals.iloc[0]:,} tokens, {totals.iloc[0]/max(ap.total_tokens,1)*100:.1f}% of prompt)")


## Section 6 — Memory failure diagnostics

Heuristic classifier for every gateway turn. Labels are not ground truth —
they're hints to narrow down which conversations are worth inspecting by hand.

Labels used:

- `thread_linkage_failure` — log row has no `conversation_id`
- `missing_scope_injection` — conversation has no `scope_type`/`scope_label`
- `history_window_too_shallow` — lots of messages in thread but tiny history tokens in prompt
- `rag_crowded_out_history` — RAG tokens dominate history tokens
- `context_block_too_large` — context section is >45% of prompt
- `token_budget_pressure` — prompt within 10% of the cap
- `missing_thread_summary` — no `context_summary` on conversation
- `unknown` — nothing matched, inspect manually

In [ ]:
classified = pmd.classify_all(data, budget=TOKEN_BUDGET_DEFAULT)
if classified.empty:
    print("(no gateway logs — skip to Section 7)")
else:
    print("Primary label counts:")
    print(classified["primary_label"].value_counts().to_string())
    print()
    display(classified.head(15))


In [ ]:
# Zoom in on the suspicious turns
if not classified.empty:
    suspicious = classified[classified["primary_label"] != "unknown"].head(10)
    for _, row in suspicious.iterrows():
        print("-" * 88)
        print(f"conversation: {row['conversation_id']}  log: {row['log_id']}")
        print(f"lane: {row['route_lane']}  prompt_tokens: {row['prompt_tokens']}")
        print(f"user: {row['user_message']}")
        print(f"labels: {row['labels']}")
        for reason in row["reasons"]:
            print(f"  · {reason}")


## Section 7 — Sensitivity experiments

Each experiment holds the `inputs` from Section 5 mostly constant and sweeps
one variable. Rerun Section 5 before any experiment if you changed the
baseline. Results are `pd.DataFrame`s so you can sort / filter / plot freely.

### Experiment A — Change page name only

Does swapping page name actually change anything the model sees? If the page
name is buried in a JSON blob, the tokenized impact may be trivial — which is
itself a finding (the model has little to distinguish pages).

In [ ]:
page_names = [
    "Fund Detail",
    "Fund Overview",
    "Portfolio",
    "Debt Surveillance",
    "AI Workbench",
    "Scenario Lab",
    "Variance",
]
pmd.experiment_page_name(inputs, page_names)


### Experiment B — Change entity label only

Degrade the label progressively to test how deictic phrases like "this fund"
get resolved. If the model only sees `fund-mvf1` and no human-readable label,
downstream answers will drift.

In [ ]:
labels = [
    "Meridian Value Fund I",
    "Meridian Value Fund",
    "Fund I",
    "this fund",
    "",
]
pmd.experiment_entity_label(inputs, labels)


### Experiment C — Change history depth

How much continuity do we lose when the history window shrinks? The
simulator's fallback trim order is RAG → oldest history → visible context →
thread entity state, so watch the `trimmed` column carefully.

In [ ]:
depths = [2, 4, 8, None]  # None = keep all history
pmd.experiment_history_depth(inputs, depths)


### Experiment D — RAG inclusion

The question: does RAG displace history? Watch `history_tokens` vs `rag_tokens`
as you move from no-RAG to full-RAG.

In [ ]:
rag_modes = {
    "none": [],
    "short": [
        "Meridian Value Fund I 2025Q4 quarter-close: TVPI 1.42, DPI 0.35, RVPI 1.07.",
    ],
    "full": [
        "Meridian Value Fund I 2025Q4 quarter-close: TVPI 1.42, DPI 0.35, RVPI 1.07.",
        "Glossary: TVPI = (cumulative distributions + NAV) / contributed capital.",
        "LP commitments: $250M, called $210M, distributed $73.5M, NAV $224.7M.",
        "Asset mix: 2 multifamily, 1 industrial; all stabilized; 2025Q4 occupancy 94%.",
        "Benchmark: peer value-add median TVPI 1.38x, peer median Net IRR 10.9%.",
    ],
}
pmd.experiment_rag_modes(inputs, rag_modes)


### Experiment E — Token cap

Simulate the prompt under progressively larger budgets. The `trimmed` column
shows what gets dropped first under each cap — that's your forensic trail when
a user says "Winston forgot the earlier part of the conversation".

In [ ]:
caps = [4000, 8000, 16000, 32000, 64000, 200000]
pmd.experiment_token_cap(inputs, caps)


### Experiment F — Role framing

Does the role framing shift the system prompt in ways that might change
phrasing? Compare the system blocks across roles.

In [ ]:
roles = ["analyst", "lp_reporting", "acquisitions", "operating_partner", "pds_executive"]
pmd.experiment_role_framing(inputs, roles)


### Experiment G — Missing / conflicting entity state

What happens when thread entity state and visible UI context disagree, or one
of them is missing entirely? The `conflict` row is the interesting one — the
gateway will ship both blocks and let the model pick, which usually degrades.

In [ ]:
pmd.experiment_entity_state(inputs)


### Experiment H — Deictic resolution

Run a set of deictic prompts ("this fund", "what is going on here?") against
a *cleanly labeled* context and then against a stripped-label context. Compare
how much risk the phrasing introduces when the scope label drops.

In [ ]:
deictic_prompts = [
    "What is going on here?",
    "Should we sell this asset?",
    "Summarize this page for an LP.",
    "Is this fund on track?",
    "What is the current environment?",
]

def deictic_row(prompt, scope_label):
    trial = pmd.ContextInputs(**asdict(inputs))
    trial.current_user_message = prompt
    trial.assistant_scope_label = scope_label
    ap = pmd.assemble_prompt(trial)
    return {
        "prompt": prompt,
        "scope_label": scope_label or "(blank)",
        "context_preview": pmd.truncate_preview(ap.context, 120),
        "total_tokens": ap.total_tokens,
        "deictic_phrases_in_prompt": pmd._deictic_count(prompt),
    }

pd.DataFrame(
    [deictic_row(p, "Meridian Value Fund I") for p in deictic_prompts] +
    [deictic_row(p, "") for p in deictic_prompts]
)


### Experiment I — Multi-turn goal retention

Add a clear goal at the start of the thread and then push several follow-ups.
Check whether the goal survives into the history block at various depths —
this catches the "the assistant forgot what I was doing" failure mode.

In [ ]:
goal_history = [
    {"role": "user", "content": "I'm validating the fund-return plumbing for 2025Q4. I will ask several follow-ups."},
    {"role": "assistant", "content": "Understood — I'll keep validation in focus."},
    {"role": "user", "content": "What is TVPI for Meridian Value Fund I?"},
    {"role": "assistant", "content": "TVPI 1.42x as of 2025Q4."},
    {"role": "user", "content": "And DPI?"},
    {"role": "assistant", "content": "DPI 0.35."},
    {"role": "user", "content": "Source for distributions?"},
    {"role": "assistant", "content": "re_capital_ledger_entry events, type=distribution."},
]

def goal_row(depth):
    trial = pmd.ContextInputs(**asdict(inputs))
    trial.prior_history_messages = goal_history
    trial.history_depth = depth
    ap = pmd.assemble_prompt(trial)
    kept = [m["content"] for m in ap.history]
    goal_retained = any("validating" in c.lower() for c in kept)
    return {
        "depth": depth if depth is not None else "all",
        "history_kept": len(ap.history),
        "goal_retained": goal_retained,
        "history_preview": pmd.truncate_preview(" | ".join(kept), 160),
    }

pd.DataFrame([goal_row(d) for d in [2, 4, 6, 8, None]])


### Experiment J — Retrieval pollution

Compare relevant vs generic vs redundant RAG snippets and watch what it does
to the `total_tokens` and to `history_tokens` under trim pressure.

In [ ]:
rag_pollution_modes = {
    "relevant": [
        "Meridian Value Fund I 2025Q4 TVPI 1.42x, Net IRR 11.8%.",
        "Meridian Value Fund I asset list: 2 multifamily, 1 industrial, 2025Q4.",
    ],
    "generic": [
        "REPE funds generally measure returns via TVPI, DPI, RVPI.",
        "A fund's Net IRR accounts for management fees and carry.",
        "Gross IRR excludes fund-level fees and expenses.",
    ],
    "redundant": [
        "TVPI 1.42x.",
        "TVPI is 1.42x as of 2025Q4.",
        "As of Q4 2025, TVPI for Meridian Value Fund I is 1.42.",
        "Meridian Value Fund I TVPI: 1.42x.",
    ],
}
pmd.experiment_rag_modes(inputs, rag_pollution_modes)


## Section 8 — Visualization

Small matplotlib-only charts. These are defensive: if a column is missing or
empty, the cell prints a notice instead of raising.

In [ ]:
logs = data.gateway_logs
if logs.empty or "prompt_tokens" not in logs.columns:
    print("(no gateway logs with prompt_tokens — skipping histogram)")
else:
    pt = pd.to_numeric(logs["prompt_tokens"], errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(pt, bins=30)
    ax.set_title("Distribution of prompt_tokens across turns")
    ax.set_xlabel("prompt_tokens")
    ax.set_ylabel("turns")
    plt.tight_layout()
    plt.show()


In [ ]:
# Stacked bar of section tokens for a handful of turns with prompt_audit_json
if logs.empty or "prompt_audit_json" not in logs.columns:
    print("(no prompt_audit_json — skipping section composition chart)")
else:
    audits = []
    for _, row in logs.head(25).iterrows():
        a = pmd.safe_json(row.get("prompt_audit_json"))
        if isinstance(a, dict) and a:
            audits.append({
                "turn": str(row.get("id", ""))[:8],
                **{k: a.get(k, 0) for k in ["system", "context", "rag", "history", "user"]},
            })
    if not audits:
        print("(no parseable prompt_audit_json rows — skipping)")
    else:
        adf = pd.DataFrame(audits).set_index("turn")
        fig, ax = plt.subplots(figsize=(9, 4))
        bottom = np.zeros(len(adf))
        for col in adf.columns:
            ax.bar(adf.index, adf[col], bottom=bottom, label=col)
            bottom = bottom + adf[col].values
        ax.set_title("Prompt section composition by turn")
        ax.set_ylabel("tokens")
        ax.legend(loc="upper right", fontsize=8)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:
# Messages per conversation
if data.messages.empty:
    print("(no messages — skipping)")
else:
    counts = data.messages.groupby("conversation_id").size()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(counts.values, bins=20)
    ax.set_title("Messages per conversation")
    ax.set_xlabel("messages")
    ax.set_ylabel("conversations")
    plt.tight_layout()
    plt.show()


In [ ]:
# % of logs lacking conversation linkage
if logs.empty:
    print("(no logs — skipping)")
else:
    linked = logs["conversation_id"].notna() & (logs["conversation_id"] != "")
    pct = linked.mean() * 100
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(["linked", "orphan"], [pct, 100 - pct])
    ax.set_ylim(0, 100)
    ax.set_ylabel("% of logs")
    ax.set_title(f"Gateway log linkage ({pct:.1f}% linked)")
    plt.tight_layout()
    plt.show()


## Section 9 — Recommendations

Summary of the top observed failure modes with candidate fixes, and a
judgment call on whether token pressure or context instability is the
primary driver. Use this as a starting point, not gospel — the classifier is
heuristic.

In [ ]:
rec = pmd.recommend(classified, data)
rec


## How to interpret results

A quick guide for reading this lab:

- **Token composition is shape, not gospel.** The estimator is `len / 4`. If a
  turn shows 60% history / 25% RAG in the audit, trust the shape; don't use
  it for billing.
- **Trim order matters more than budget.** If the `trimmed` column shows
  `rag, history_oldest`, the model lost continuity even though the prompt fit.
- **Labels win.** Experiment B almost always shows that swapping a human
  label for an id or a blank makes deictic prompts ("this fund") ambiguous.
  This is the single cheapest fix for phrasing drift.
- **Orphan logs (no `conversation_id`) are the #1 memory bug.** If Section 9
  shows a non-trivial `thread_linkage_failure` count, fix that before
  touching prompt composition.
- **Pay attention to `context_block_too_large`.** A context block past ~45%
  of prompt is usually dumping JSON that could be a one-line summary.
- **Missing `context_summary` on a conversation is a latent memory bomb.**
  If the UI context ever drops, the model has nothing to fall back on.

When you find something worth fixing, capture the `conversation_id` and the
experiment cell that reproduced it — those two things together are enough to
file a ticket or hand off to the `winston-remediation-playbook` skill.